# 02 — 数据清洗、变量构造与面板合并

本 Notebook 完成以下任务：
1. 读取 7 个原始文件，合并跨时段表
2. 过滤「报表类型(合并报表)」等元数据行
3. 标准化标识符（code、year）
4. 构造 13 个财务变量
5. 检查每个变量的实际可用年份
6. 更新变量字典（含公式和起始年份）
7. 缩尾处理 → 输出最终面板数据集

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

BASE = Path.cwd()
RAW_DIR = BASE / 'data' / 'raw'
CLEAN_DIR = BASE / 'data' / 'clean'
COMBINED_DIR = BASE / 'data' / 'combined'
DICT_DIR = BASE / 'data' / 'dict'
OUTPUT_TABLES = BASE / 'output' / 'tables'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

In [3]:
import textwrap, os
log_path = BASE / 'process_log.txt'
def log(msg):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {msg}'
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line + '\n')
    print(line)

# Create fresh log
with open(log_path, 'w', encoding='utf-8') as f:
    f.write(f'=== Process Log: 02_clean_construct_variables.ipynb ===\n')
    f.write(f'Started at: {TS}\n\n')
log('Notebook 2 started')

[2026-05-20 22:22:43] Notebook 2 started


---
## 1. 读取原始数据

### 1.1 资产负债表（合并两时段）

In [4]:
bs_early = pd.read_excel(RAW_DIR / '资产负债表-2000-2010.xlsx', skiprows=[1,2])
bs_late  = pd.read_excel(RAW_DIR / '资产负债表-2011-2024.xlsx', skiprows=[1,2])

# Confirm identical columns
assert list(bs_early.columns) == list(bs_late.columns), 'BS columns differ across periods'

bs_all = pd.concat([bs_early, bs_late], ignore_index=True)
log(f'资产负债表: early={len(bs_early)} rows, late={len(bs_late)} rows, merged={len(bs_all)} rows')

# Filter "报表类型(合并报表)" metadata rows
before = len(bs_all)
bs_all = bs_all[bs_all['code'].notna()].copy()
log(f'  过滤元数据行: {before} → {len(bs_all)} rows')

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[2026-05-20 22:24:39] 资产负债表: early=64164 rows, late=81663 rows, merged=145827 rows
[2026-05-20 22:24:39]   过滤元数据行: 145827 → 145825 rows


### 1.2 利润表-现金流量表（合并两时段）

In [5]:
is_early = pd.read_excel(RAW_DIR / '利润表-现金流量表-2000-2010.xlsx', skiprows=[1,2])
is_late  = pd.read_excel(RAW_DIR / '利润表-现金流量表-2011-2024.xlsx', skiprows=[1,2])

assert list(is_early.columns) == list(is_late.columns), 'IS columns differ across periods'

is_all = pd.concat([is_early, is_late], ignore_index=True)
log(f'利润表: early={len(is_early)} rows, late={len(is_late)} rows, merged={len(is_all)} rows')

before = len(is_all)
is_all = is_all[is_all['code'].notna()].copy()
log(f'  过滤元数据行: {before} → {len(is_all)} rows')

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[2026-05-20 22:27:31] 利润表: early=64164 rows, late=81663 rows, merged=145827 rows
[2026-05-20 22:27:32]   过滤元数据行: 145827 → 145825 rows


### 1.3 CSMAR 常用变量（股权结构、市值等）

In [6]:
cc = pd.read_excel(RAW_DIR / 'CSMAR常用变量-2000-2024.xlsx', skiprows=[1,2])
log(f'CSMAR常用变量: {len(cc)} rows, {len(cc.columns)} cols')

# Rename Stkcd → code, accper → year for merging
cc = cc.rename(columns={'Stkcd': 'code', 'accper': 'year'})
cc['code'] = cc['code'].astype(float)

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[2026-05-20 22:28:26] CSMAR常用变量: 61456 rows, 33 cols


### 1.4 上市公司基本信息年度表（上市日期用于 Age）

In [7]:
info = pd.read_excel(RAW_DIR / '上市公司基本信息年度表.xlsx', skiprows=[1,2])
log(f'基本信息年度表: {len(info)} rows')

# Keep only core columns for Age construction
age_cols = ['Symbol', 'EndDate', 'LISTINGDATE', 'ShortName']
info_age = info[age_cols].dropna(subset=['LISTINGDATE']).copy()
info_age = info_age.rename(columns={'Symbol': 'code'})
info_age['code'] = info_age['code'].astype(float)
info_age['listing_year'] = pd.to_datetime(info_age['LISTINGDATE']).dt.year

# One firm may appear multiple times; keep unique code → listing_year
info_age = info_age.drop_duplicates(subset=['code', 'listing_year']).copy()
# For each code, take the earliest listing year
info_age = info_age.groupby('code', as_index=False)['listing_year'].min()
log(f'  上市年份映射: {len(info_age)} unique firms')

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[2026-05-20 22:30:12] 基本信息年度表: 64171 rows
[2026-05-20 22:30:12]   上市年份映射: 5613 unique firms


---
## 2. 清洗各表

In [8]:
def clean_table(df, code_col='code', date_col='EndDate'):
    """Standardize code and year across all tables."""
    df = df.copy()
    # Ensure code is float (for cross-table merge)
    df[code_col] = df[code_col].astype(float)
    # Convert EndDate to integer year
    df['year'] = df[date_col].astype(int)
    return df

bs_all = clean_table(bs_all)
is_all = clean_table(is_all)
log(f'清洗完成: BS={len(bs_all)}, IS={len(is_all)}, CC={len(cc)}')

[2026-05-20 22:30:13] 清洗完成: BS=145825, IS=145825, CC=61456


In [9]:
# Check for duplicates per code-year
for name, df in [('BS', bs_all), ('IS', is_all), ('CC', cc)]:
    dups = df.duplicated(subset=['code', 'year'], keep=False).sum()
    if dups:
        log(f'  {name}: {dups} duplicate code-year rows (keeping first)')
        df.drop_duplicates(subset=['code', 'year'], keep='first', inplace=True)
    else:
        log(f'  {name}: no duplicates')

[2026-05-20 22:30:13]   BS: no duplicates
[2026-05-20 22:30:13]   IS: no duplicates
[2026-05-20 22:30:13]   CC: no duplicates


In [10]:
# Save cleaned individual tables
bs_all.to_csv(CLEAN_DIR / 'balance_sheet_clean.csv', index=False, encoding='utf-8-sig')
is_all.to_csv(CLEAN_DIR / 'income_statement_clean.csv', index=False, encoding='utf-8-sig')
cc.to_csv(CLEAN_DIR / 'csmar_common_clean.csv', index=False, encoding='utf-8-sig')
info_age.to_csv(CLEAN_DIR / 'listing_info_clean.csv', index=False, encoding='utf-8-sig')
log('Cleaned tables saved to data/clean/')

[2026-05-20 22:30:23] Cleaned tables saved to data/clean/


---
## 3. 合并为面板数据

以资产负债表为主表，依次左连接利润表、CSMAR 常用变量和上市日期信息。

In [11]:
# Select only the columns we need from each table
bs_cols = ['code', 'year', 'stknme',
           'FS_Combas-A001101000',  # 货币资金
           'FS_Combas-A001000000',  # 资产总计
           'FS_Combas-A002000000',  # 负债合计
           'FS_Combas-A002100000',  # 流动负债合计
           'FS_Combas-A002200000',  # 非流动负债合计
           'FS_Combas-A002101000',  # 短期借款
           'FS_Combas-A002201000',  # 长期借款
           'FS_Combas-A003000000']  # 所有者权益合计

is_cols = ['code', 'year',
           'FS_Comins-B002000000']  # 净利润

cc_cols = ['code', 'year',
           'Shrcr1',     # Top1
           'Shrhfd5']    # HHI5

bs_merge = bs_all[bs_cols].copy()
is_merge = is_all[is_cols].copy()
cc_merge = cc[cc_cols].copy()

# Merge step by step
panel = bs_merge.merge(is_merge, on=['code', 'year'], how='left', suffixes=('', '_is'))
panel = panel.merge(cc_merge, on=['code', 'year'], how='left', suffixes=('', '_cc'))
panel = panel.merge(info_age, on='code', how='left', suffixes=('', '_info'))

log(f'面板数据合并完成: {len(panel)} rows, {len(panel.columns)} cols')
panel.to_csv(COMBINED_DIR / 'panel_merged_raw.csv', index=False, encoding='utf-8-sig')
log('panel 已保存到 disk，后续可跳过前序步骤直接加载')

[2026-05-20 22:30:24] 面板数据合并完成: 145825 rows, 15 cols
[2026-05-20 22:30:26] panel 已保存到 disk，后续可跳过前序步骤直接加载


In [12]:
# Check merge coverage
for col in ['FS_Comins-B002000000', 'Shrcr1', 'Shrhfd5', 'listing_year']:
    if col not in panel.columns:
        log(f'  {col}: column not found')
        continue
    pct = panel[col].notna().mean() * 100
    log(f'  {col}: {pct:.1f}% non-missing')

[2026-05-20 22:30:26]   FS_Comins-B002000000: 49.2% non-missing
[2026-05-20 22:30:26]   Shrcr1: 39.6% non-missing
[2026-05-20 22:30:26]   Shrhfd5: 39.6% non-missing
[2026-05-20 22:30:26]   listing_year: 96.1% non-missing


---
## 4. 构造 13 个变量

每个变量均检查实际可用年份，并在变量字典的 `note` 中记录公式。

In [14]:
# 尝试从内存获取 panel，否则从磁盘加载
PANEL_CSV = COMBINED_DIR / 'panel_merged_raw.csv'
try:
    df = panel.copy()
except NameError:
    if PANEL_CSV.exists():
        log('panel not in memory, reloading from disk...')
        panel = pd.read_csv(PANEL_CSV, encoding='utf-8-sig')
        df = panel.copy()
    else:
        raise RuntimeError('panel not in memory and no saved CSV. '
                           'Please run cells 19-26 first to create the merged dataset.')

# Rename CSMAR codes to readable names for clarity
col_map = {
    'FS_Combas-A001101000': 'cash',
    'FS_Combas-A001000000': 'total_assets',
    'FS_Combas-A002000000': 'total_liab',
    'FS_Combas-A002100000': 'current_liab',
    'FS_Combas-A002200000': 'noncurrent_liab',
    'FS_Combas-A002101000': 'short_loan',
    'FS_Combas-A002201000': 'long_loan',
    'FS_Combas-A003000000': 'equity',
    'FS_Comins-B002000000': 'net_income',
}
df = df.rename(columns=col_map)

# 确保财务列是数值类型（从 xlsx/CSV 读入后可能为 object）
num_cols = ['total_assets', 'total_liab', 'current_liab', 'noncurrent_liab',
            'cash', 'short_loan', 'long_loan', 'equity', 'net_income',
            'Shrcr1', 'Shrhfd5', 'listing_year']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 关键修复：过滤掉总资产<=0的行，防止后续除零
n_before = len(df)
df = df[df['total_assets'] > 0].copy()
n_dropped = n_before - len(df)
if n_dropped:
    log(f'  Dropped {n_dropped} rows with total_assets <= 0 (or NaN)')

log(f'Column rename complete — {len(df)} rows ready for variable construction')

[2026-05-20 22:33:10]   Dropped 74038 rows with total_assets <= 0 (or NaN)
[2026-05-20 22:33:10] Column rename complete — 71787 rows ready for variable construction


In [15]:
def first_year(series):
    """Find the first year with non-missing data for a variable.
    Returns int year, or 'N/A' if no valid data."""
    valid = df.loc[series.notna(), 'year']
    return int(valid.min()) if len(valid) else 'N/A'

var_notes = {}

In [16]:
# Lev = 总负债 / 总资产
df['Lev'] = df['total_liab'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['Lev'])
var_notes['Lev'] = f'Lev = total_liab / total_assets; available from {fy}'
log(f'Lev constructed, available from {fy}, mean={df["Lev"].mean():.4f}')

[2026-05-20 22:33:17] Lev constructed, available from 2000, mean=0.5006


In [17]:
# SL = 流动负债 / 总资产
df['SL'] = df['current_liab'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['SL'])
var_notes['SL'] = f'SL = current_liab / total_assets; available from {fy}'
log(f'SL constructed, available from {fy}, mean={df["SL"].mean():.4f}')

[2026-05-20 22:33:19] SL constructed, available from 2000, mean=0.3975


In [18]:
# LL = 非流动负债 / 总资产
df['LL'] = df['noncurrent_liab'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['LL'])
var_notes['LL'] = f'LL = noncurrent_liab / total_assets; available from {fy}'
log(f'LL constructed, available from {fy}, mean={df["LL"].mean():.4f}')

[2026-05-20 22:33:21] LL constructed, available from 2000, mean=0.0979


In [19]:
# SDR = 流动负债 / 总负债
df['SDR'] = df['current_liab'] / df['total_liab'].replace(0, np.nan)
fy = first_year(df['SDR'])
var_notes['SDR'] = f'SDR = current_liab / total_liab; available from {fy}'
log(f'SDR constructed, available from {fy}, mean={df["SDR"].mean():.4f}')

[2026-05-20 22:33:22] SDR constructed, available from 2000, mean=0.8210


In [20]:
# Cash = 货币资金 / 总资产
df['Cash'] = df['cash'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['Cash'])
var_notes['Cash'] = f'Cash = cash_M0 / total_assets; available from {fy}'
log(f'Cash constructed, available from {fy}, mean={df["Cash"].mean():.4f}')

[2026-05-20 22:33:24] Cash constructed, available from 2000, mean=0.1856


In [21]:
# ROA = 净利润 / 总资产
df['ROA'] = df['net_income'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['ROA'])
var_notes['ROA'] = f'ROA = net_income / total_assets; available from {fy}'
log(f'ROA constructed, available from {fy}, mean={df["ROA"].mean():.4f}')

[2026-05-20 22:33:26] ROA constructed, available from 2000, mean=0.3324


In [22]:
# ROE = 净利润 / 所有者权益
df['ROE'] = df['net_income'] / df['equity'].replace(0, np.nan)
fy = first_year(df['ROE'])
var_notes['ROE'] = f'ROE = net_income / equity; available from {fy}'
log(f'ROE constructed, available from {fy}, mean={df["ROE"].mean():.4f}')

[2026-05-20 22:33:28] ROE constructed, available from 2000, mean=0.0790


In [24]:
# SLoan = 短期借款 / 总资产
df['SLoan'] = df['short_loan'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['SLoan'])
var_notes['SLoan'] = f'SLoan = short_loan / total_assets; available from {fy}'
log(f'SLoan constructed, available from {fy}, mean={df["SLoan"].mean():.4f}')

[2026-05-20 22:33:32] SLoan constructed, available from 2000, mean=0.1305


In [26]:
# LLoan = 长期借款 / 总资产
df['LLoan'] = df['long_loan'] / df['total_assets'].replace(0, np.nan)
fy = first_year(df['LLoan'])
var_notes['LLoan'] = f'LLoan = long_loan / total_assets; available from {fy}'
log(f'LLoan constructed, available from {fy}, mean={df["LLoan"].mean():.4f}')

[2026-05-20 22:33:36] LLoan constructed, available from 2000, mean=0.0623


In [27]:
# Top1 = 第一大股东持股比例
df['Top1'] = df['Shrcr1']
fy = first_year(df['Top1'])
var_notes['Top1'] = f'Top1 = Shrcr1 (direct); available from {fy}'
log(f'Top1 constructed, available from {fy}, mean={df["Top1"].mean():.4f}')

[2026-05-20 22:33:38] Top1 constructed, available from 2003, mean=34.3953


In [28]:
# HHI5 = 前五大股东持股集中度
df['HHI5'] = df['Shrhfd5']
fy = first_year(df['HHI5'])
var_notes['HHI5'] = f'HHI5 = Shrhfd5 (direct); available from {fy}'
log(f'HHI5 constructed, available from {fy}, mean={df["HHI5"].mean():.4f}')

[2026-05-20 22:33:40] HHI5 constructed, available from 2003, mean=0.1625


In [29]:
# Size = ln(总资产)
df['Size'] = np.log(df['total_assets'].clip(lower=1e-10))
fy = first_year(df['Size'])
var_notes['Size'] = f'Size = ln(total_assets); available from {fy}'
log(f'Size constructed, available from {fy}, mean={df["Size"].mean():.4f}')

[2026-05-20 22:33:41] Size constructed, available from 2000, mean=21.9730


In [30]:
# Age = year - 上市年份 + 1
df['Age'] = df['year'] - df['listing_year'] + 1
df.loc[df['Age'] < 0, 'Age'] = np.nan  # listing after current year → invalid
fy = first_year(df['Age'])
var_notes['Age'] = f'Age = year - listing_year + 1; available from {fy}'
log(f'Age constructed, available from {fy}, mean={df["Age"].mean():.1f}')

[2026-05-20 22:33:43] Age constructed, available from 2000, mean=10.4


### 4.1 更新变量字典

In [31]:
# Load existing dictionary
var_dict = pd.read_csv(DICT_DIR / 'variable_dictionary.csv', encoding='utf-8-sig')

# Update clean_variable and note for each constructed variable
update_map = {
    'A001000000': 'total_assets',
    'A002000000': 'total_liab',
    'A002100000': 'current_liab',
    'A002200000': 'noncurrent_liab',
    'A001101000': 'cash',
    'A002101000': 'short_loan',
    'A002201000': 'long_loan',
    'A003000000': 'equity',
    'B002000000': 'net_income',
    'Shrcr1': 'Top1',
    'Shrhfd5': 'HHI5',
}

for code, clean_name in update_map.items():
    mask = var_dict['raw_variable'].str.contains(code, na=False)
    var_dict.loc[mask, 'clean_variable'] = clean_name

# Add 13 constructed variables as new rows
new_rows = []
for varname in ['Lev', 'SL', 'LL', 'SDR', 'Cash', 'ROA', 'ROE', 'SLoan', 'LLoan', 'Top1', 'HHI5', 'Size', 'Age']:
    note = var_notes.get(varname, '')
    new_rows.append({
        'source_file': '02_clean_construct_variables.ipynb (constructed)',
        'raw_variable': varname,
        'clean_variable': varname,
        'definition': varname,
        'unit': 'ratio' if varname not in ['Size', 'Age'] else ('ln(yuan)' if varname == 'Size' else 'years'),
        'note': note
    })

var_dict = pd.concat([var_dict, pd.DataFrame(new_rows)], ignore_index=True)
var_dict.to_csv(DICT_DIR / 'variable_dictionary.csv', index=False, encoding='utf-8-sig')
log(f'变量字典更新: {len(var_dict)} rows (was 218, +13 constructed)')

[2026-05-20 22:33:45] 变量字典更新: 256 rows (was 218, +13 constructed)


---
## 5. 缩尾处理（Winsorize）

对比例型变量按年度进行 1%/99% 缩尾。

In [32]:
def winsorize_group(grp, cols, pct=0.01):
    """Winsorize specified columns at given percentile within a group."""
    grp = grp.copy()
    for col in cols:
        lo = grp[col].quantile(pct)
        hi = grp[col].quantile(1 - pct)
        grp[col] = grp[col].clip(lo, hi)
    return grp

winsorize_cols = ['Lev', 'SL', 'LL', 'SDR', 'Cash', 'ROA', 'ROE', 'SLoan', 'LLoan', 'HHI5']
_years = df['year']
df = df.groupby('year', group_keys=False).apply(winsorize_group, winsorize_cols, include_groups=False)
df['year'] = _years
log(f'Winsorized {len(winsorize_cols)} variables at 1%/99% by year')

[2026-05-20 22:33:50] Winsorized 10 variables at 1%/99% by year


---
## 6. 保存最终面板数据

In [33]:
# Final column selection
final_cols = ['code', 'year', 'stknme',
              'Lev', 'SL', 'LL', 'SDR', 'Cash', 'ROA', 'ROE',
              'SLoan', 'LLoan', 'Top1', 'HHI5', 'Size', 'Age']
final = df[final_cols].copy()

# Sort
final = final.sort_values(['code', 'year']).reset_index(drop=True)

# Save CSV
final.to_csv(COMBINED_DIR / 'csmar_firm_year_panel.csv', index=False, encoding='utf-8-sig')
log(f'最终面板: {len(final)} firm-year obs, {len(final.columns)} vars')
log(f'  年份范围: {int(final["year"].min())}–{int(final["year"].max())}')
log(f'  公司数量: {final["code"].nunique()}')

[2026-05-20 22:33:54] 最终面板: 71787 firm-year obs, 16 vars
[2026-05-20 22:33:54]   年份范围: 2000–2024
[2026-05-20 22:33:54]   公司数量: 5820


In [34]:
# Quick summary of all 13 variables
summary = final.describe().T[['count', 'mean', 'std', 'min', 'max']]
summary.to_csv(OUTPUT_TABLES / 'variable_summary.csv', encoding='utf-8-sig')
print(summary.round(4).to_string())

         count         mean          std        min          max
code   71787.0  347021.3982  292273.1609     1.0000  920819.0000
year   71787.0    2015.4916       6.6976  2000.0000    2024.0000
Lev    71787.0       0.4523       0.2611     0.0279       3.9719
SL     70819.0       0.3595       0.2185     0.0000       3.1512
LL     69996.0       0.0858       0.1070    -0.0191       1.0434
SDR    70818.0       0.8215       0.1825     0.0000       1.0493
Cash   71458.0       0.1848       0.1390     0.0005       0.8524
ROA    71787.0       0.0293       0.0847    -1.0173       0.4132
ROE    71784.0       0.0438       0.2313    -2.7131       1.8415
SLoan  62635.0       0.1159       0.1207     0.0000       1.2798
LLoan  52544.0       0.0613       0.0842     0.0000       0.4674
Top1   57744.0      34.3953      15.3839     0.2863      89.9910
HHI5   57744.0       0.1618       0.1175     0.0107       0.6127
Size   71787.0      21.9730       1.5585    10.8422      31.5192
Age    67687.0      10.41

In [35]:
log('Notebook 2 completed successfully')
print()
print('===== 待执行清单 =====')
print('[x] 读取 7 个 xlsx，合并跨时段文件')
print('[x] 过滤"报表类型(合并报表)"常量行')
print('[x] 构造 13 个变量: Lev, SL, LL, SDR, Cash, ROA, ROE, SLoan, LLoan, Top1, HHI5, Size, Age')
print('[x] 合并为最终面板数据集, 存入 data/combined/')
print('[x] 更新变量字典 (含公式和起始年份)')
print('[x] process_log.txt 同步更新')

[2026-05-20 22:33:58] Notebook 2 completed successfully

===== 待执行清单 =====
[x] 读取 7 个 xlsx，合并跨时段文件
[x] 过滤"报表类型(合并报表)"常量行
[x] 构造 13 个变量: Lev, SL, LL, SDR, Cash, ROA, ROE, SLoan, LLoan, Top1, HHI5, Size, Age
[x] 合并为最终面板数据集, 存入 data/combined/
[x] 更新变量字典 (含公式和起始年份)
[x] process_log.txt 同步更新
